# How the graph is built: criterion x quantile

The dial notebooks move what the receptor vector is *mixed from*. This one moves
**which edges exist at all**. The message-passing graph keeps only K train molecules:
the quantile sets how many survive, the criterion picks which ones
(`orbind/mol_selection.py`).

Every number the paper reports stands on ONE point of this grid. The figures below
exist to say whether that point is defensible, not merely inherited -- so the point
itself is marked on every panel.

**This notebook draws and does not aggregate.** Fold means, intervals and paired
differences all come from `scripts/article_sweeps/s4_quantile_grid.py`, which delegates
the averaging rule to `alpha_grid`: model seeds are averaged *inside* each fold, and
the interval is over folds. A figure that re-derived its own means would treat the
(fold, seed) cells as independent observations and halve every band on the page.

Produced by `scripts/article_sweeps/s4_run_quantile_criteria.py`.

In [ ]:
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea, VPacker

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "article_figures"))

import figkit as fk
from scripts.article_sweeps import s4_quantile_grid as qg

fk.use_house_style()
print("repo:", ROOT)

## Knobs

In [ ]:
# ================================== KNOBS ====================================
ROOT_DIR   = "results/article_sweeps/quantile_criteria"
DATASET    = None          # None -> everything on disk; or "m2or" / ["cc", "hc"]
REGIME     = None          # None -> both; or "transductive" / "inductive"
MOL_SOURCE = None          # None -> whatever is there; or "chemberta"
GNN_REGIME = None          # which MODEL wrote the rows, e.g. "sampled25-10+norm".
                           # None -> whatever is there, and the guard below
                           # refuses a frame holding two
COMBO      = "cls+mol"     # the head; the reference arm is kept whatever this says
SPLIT      = "val"         # where a quantile may be CHOSEN. Decided 25.09:
                           # this page works on validation and `test` is read
                           # ONCE, at the end, for the cell already picked

SEEDS      = None          # None -> every seed on disk. Name them to cut back to
                           # a COMPLETE grid when a seed was added and its run
                           # has not finished -- see the balance check below
CI_LEVEL   = 0.95          # Student-t over FOLDS (seeds averaged inside each fold)
METRICS    = None          # None -> the metric of record per dataset
HIGHLIGHT  = None          # None -> all eight in their own hue, which is what the
                           # palette was sized for and what a CRITERION comparison
                           # needs. "auto" dims all but the reported one -- right
                           # when the figure's job is "ours against the rest"
SHOW_K     = True          # the K panel: what the quantile actually cuts

SAVE_FIGS  = True
FIG_DIR    = "results/article_figures/quantile_criteria"
FIG_FORMATS = ("png", "pdf")   # vector for print, PNG for looking at it here

# ------------------- the headline pair (25.09), in the dial's idiom ----------
# The same knobs as `prediction_dial`'s highlight figure, and for the same reasons:
# built at the FINAL printed width, because fonts do not scale when a figure is shrunk
# into a column.
PANELS     = [("m2or", "transductive"), ("m2or", "inductive")]
FIG_WIDTH_IN = 7.1     # full page width (180 mm): two panels of 3.55 in
ASPECT     = 1.00      # height / width of the FIGURE cell, NOT of the axes. The
                       # title, the x label, the rotated ticks and the legend strip all
                       # come out of it, so the axes end up flatter than this number.
                       # Measured for the present layout: 0.90 -> axes 0.62, 1.00 ->
                       # axes 0.75 (what we want), 1.15 -> axes 0.94. Re-measure after
                       # anything that changes the overhead, e.g. a legend row
BAND_CRITS = ()        # whose 95% band is DRAWN. () for none, ("greedy_pair_cover",)
                       # for the reported construction alone, tuple(fk.CRIT) for all
                       # -- eight of them is grey mud
BAND_ALPHA = 0.13
SHOW_FLOOR = False     # the I-mark in the left margin: the smallest difference this
                       # grid resolves. True to bring the ruler back
HEADROOM   = 0.0       # extra y range on the box's side. Only the box needs it, and
                       # without one it just flattens the curves
BOX_CORNER = "upper left"   # ONE corner in both panels: two boxes in two places read
                       # as two decisions
SHOW_BOX   = False     # the panel's own summary (best cell, where ours stands). It is
                       # in the `sep`/`behind` tables below either way
SHOW_RING  = True      # ring the cell the paper reports
PANEL_LETTERS = True
LETTER_OFFSET = (-22, 1)   # points from the axes' top-left corner: left and up
TITLE_PAD  = 13
TITLE_FONT, TICK_FONT, BOX_FONT = 9.0, 8.0, 7.0
XTICK_FONT, XLABEL_FONT, YLABEL_FONT = 6.5, 8.0, 8.5
LEGEND_NCOL, LEGEND_PAD_IN, LEGEND_FONT = 6, 0.38, 6.8   # 11 entries: 6 keeps it
                       # to two rows, and every row of legend is height taken from
                       # the panels
DS_LABEL   = {"m2or": "M2OR", "cc": "Mosquito", "hc": "Fly"}   # the paper's names
RG_LABEL   = {"transductive": "Seen molecules", "inductive": "Cold molecules"}
#: what the legend calls each criterion -- the names the appendix uses
CRIT_LABEL = {"coverage": "coverage", "balance_bits": "balance",
              "entropy_bits": "entropy", "disc_pairs": "discriminative pairs",
              "idf_coverage": "IDF coverage", "composite": "composite",
              "greedy_pair_cover": "greedy pair cover", "random": "random"}
Y_LABEL    = r"$\Delta$ vs. XGBoost-base"
# =============================================================================

df = qg.load(root=ROOT_DIR, dataset=DATASET, regime=REGIME, mol_source=MOL_SOURCE,
             combo=COMBO, split=SPLIT, seeds=SEEDS, gnn_regime=GNN_REGIME)

SERIES = sorted(df.series.unique())
QS = sorted(pd.to_numeric(qg.graph_rows(df)["quantile"], errors="coerce").dropna().unique())
CRITS = [c for c in fk.CRIT if c in set(qg.graph_rows(df).criterion)]

In [ ]:
# ---------------------------------------------------------------- what is on disk
# One guard, and it matters: `k_mode` decides what a quantile MEANS. Under
# `coverage_quantile` q cuts on the coverage distribution; under `fraction` it keeps
# the top (1-q) share outright. On a complete matrix the first keeps everything at
# every q. A frame holding both puts two different knobs on one x axis.
modes = sorted(df.k_mode.astype(str).unique())
if len(modes) != 1:
    raise SystemExit(f"this frame holds k_mode {modes} -- two knobs cannot share an "
                     f"axis. Filter to one, or read them as separate figures.")
K_MODE = modes[0]

# The same guard for the MODEL. The out directory is resumable, so an older run left in
# it is concatenated with the reported one without a word, and the curves then mix two
# encoders. The q<=0 column is where this shows first: the sweep shares that cell, so
# eight identical numbers are guaranteed WITHIN a run and impossible ACROSS two.
PROV = qg.provenance(df)
if len(PROV) > 1:
    display(PROV)
    raise SystemExit("this frame holds more than one run (see the table above). Set "
                     "GNN_REGIME, or move the older CSVs out of ROOT_DIR.")

print(f"{len(SERIES)} series | {len(QS)} quantiles | {len(CRITS)} criteria | "
      f"k_mode: {K_MODE} | split: {SPLIT}")
print("quantiles:", ", ".join(f"{q:g}" for q in QS))
print("files:", ", ".join(pathlib.Path(f).name for f in df.attrs["files"]))

# The second guard: are the seeds the same everywhere? Adding a seed is safe and
# the sweep picks it up, but a run killed part-way through one leaves some cells
# averaging more draws than their neighbours, and the curve then moves by the
# imbalance instead of by the knob. Cut back with SEEDS until this says balanced.
bal = qg.seed_balance(df)
if len(bal) and not bal["balanced"].all():
    print("\nWARNING: unbalanced seeds -- some cells carry fewer than the rest.\n"
          "         Set SEEDS to the seeds that are COMPLETE, or finish the run.")
display(bal)

# `shared` counts rows copied rather than fitted: at q<=0 nothing is cut, so every
# criterion is the same graph there and one fit stands for all of them.
qg.coverage(df)

In [ ]:
MOF = qg.metric_for(df)
DS_OF = dict(zip(df.series, df.dataset))
print("metric of record:", MOF)

# How small a difference this grid can resolve at all: the model noise LEFT IN A FOLD
# MEAN after the seeds are averaged. Every curve below is made of fold means, so a
# wiggle shorter than this is not a finding, whatever the band says. With one seed it
# is undefined -- and then no I-mark is drawn, which is the honest statement.
FLOOR = {}
for s in SERIES:
    f = qg.floor(df[df.series == s], MOF[DS_OF[s]])
    if len(f):
        FLOOR[s] = float(f.iloc[0]["floor"])
if FLOOR:
    display(pd.Series(FLOOR, name="resolution floor").round(4))
else:
    print("one model seed per cell -- no resolution floor; re-run with --seeds 42 43 ...")

---
## 1. The figure

Two panels, M2OR's two regimes. Every curve is the **paired** advantage of one
construction over the boosting base, taken fold by fold, so the base is the zero line
rather than a curve of its own and every interval is the paired one.

The question the panel answers is not "which cell is best" -- with a surface this flat
the argmax is a ranking of noise -- but whether the cell we report is defensible. So
the box names the best cell on the panel, how far the next cell sits behind it in the
best cell's own half-widths, and where ours stands by the same measure. `random` is
the control: a ranked criterion that does not clear it buys message passing, not a
choice of hubs.

In [ ]:
# ========================== THE HEADLINE PAIR ================================
# Eight curves on one axis is the most a colour-vision-safe palette can carry, so each
# criterion has its own MARKER as well as its own hue, and `random` is grey -- the
# control the rest are read against, not a ninth competitor. Only the reported
# construction keeps a band: eight of them is grey mud. The I-mark in the left margin
# is what this grid can resolve at all, and a wiggle shorter than it is not a finding
# whatever the band says.
METRIC_TEX = {"R2": "$R^2$"}
LETTERS = "abcdefghijklmnop"
#: the glyph that matches each criterion's marker, for the box
GLYPH = {"o": "\u25cf", "s": "\u25a0", "^": "\u25b2", "D": "\u25c6", "v": "\u25bc",
         "P": "\u271a", "*": "\u2605", "x": "\u2715"}


def highlight_for(series):
    """Dimming is figure-wide or it is nothing: a key showing eight hues beside a plot
    drawn in grey is worse than no key. When the panels report different criteria,
    nothing is dimmed."""
    if HIGHLIGHT != "auto":
        return HIGHLIGHT
    return qg.PAPER_POINT.get(DS_OF.get(series), (None, None))[0]


_wanted = {highlight_for(s) for s in SERIES}
HL = _wanted.pop() if len(_wanted) == 1 else None
print("highlighted:", HL or "none -- all criteria in their own hue")


def draw_panel(a, ds, regime):
    """One panel: every criterion's paired advantage over the base, against q.

    Returns the panel's summary (best cell, the runner-up behind it, where ours sits),
    or {} if nothing for this cell is on disk."""
    s = f"{ds} / {regime}"
    metric = MOF.get(ds)
    sub = df[df.series == s]
    a.axhline(0.0, color=fk.C["boost_full"], lw=1.3, ls=fk.DASH["boost_full"],
              zorder=2.5)
    if sub.empty or metric is None:
        fk.note_empty(a)
        return {}
    d = qg.delta_vs_boost(sub, metric, level=CI_LEVEL)
    if d.empty:
        fk.note_empty(a)
        return {}
    for crit in CRITS:
        q = d[d.criterion == crit].sort_values("quantile")
        if q.empty:
            continue
        col, mk, lw, z = fk.crit_style(crit, HL)
        if crit in (BAND_CRITS or ()):
            fk.band(a, q["quantile"], q["lo"], q["hi"], col, BAND_ALPHA)
        a.plot(q["quantile"], q["mean"], color=col, marker=mk, lw=lw, zorder=z,
               markersize=4.0)
    # q is how much is CUT, not how much is kept: under `coverage_quantile` a molecule
    # survives only if its coverage reaches the q-th quantile, and under `fraction` the
    # top (1-q) share survives. Either way the axis runs from the whole graph on the
    # left to a handful of hubs on the right, and a label that does not say so invites
    # the figure to be read backwards.
    fk.knob_axis(a, QS, label="cut below the $q$-th coverage quantile"
                 if K_MODE == "coverage_quantile" else "cut all but the top $1-q$")
    a.tick_params(axis="y", labelsize=TICK_FONT)
    a.tick_params(axis="x", labelsize=XTICK_FONT)
    a.xaxis.label.set_size(XLABEL_FONT)

    # The ruler, when drawn, goes in a margin of its own, left of q=0: inside the plot
    # it lands on the very curves it is there to qualify.
    x0, x1 = a.get_xlim()
    if SHOW_FLOOR:
        a.set_xlim(x0 - 0.07, x1)
    # A ruler longer than the thing it measures is clipped by the axes and then reads
    # as a spine rather than as a mark, so it is said in words instead of drawn.
    _fl, (_y0, _y1) = FLOOR.get(s), a.get_ylim()
    if SHOW_FLOOR and _fl and 2 * _fl > 0.8 * (_y1 - _y0):
        print(f"NOTE {s}: the resolution floor ({_fl:.4f}) is as large as the panel -- "
              f"nothing on it is resolvable at this seed count, so no I-mark is drawn")
    elif SHOW_FLOOR:
        fk.resolution_mark(a, _fl, x=x0 - 0.04)

    b = qg.best_cell(d, metric, paper=qg.PAPER_POINT.get(ds))
    if SHOW_RING and b.get("in_grid") and np.isfinite(b.get("ours_mean", np.nan)):
        a.plot([b["ours_q"]], [b["ours_mean"]], marker="o", markersize=11,
               markerfacecolor="none", markeredgecolor=fk.INK, markeredgewidth=1.2,
               zorder=4)
    if HEADROOM:
        y0, y1 = a.get_ylim()
        pad = (y1 - y0) * HEADROOM
        a.set_ylim(y0 - (pad if BOX_CORNER.startswith("lower") else 0),
                   y1 + (pad if BOX_CORNER.startswith("upper") else 0))
    if SHOW_BOX and b:
        # The criterion is named by the leading GLYPH, not spelled out: the longest name
        # is 17 characters and the box has to fit inside a 3.5 in panel.
        lines = [(b["best_criterion"],
                  f"best q={b['best_q']:g}  {b['best_mean']:+.3f}"
                  + (f"  next {b['sep']:.1f} hw" if np.isfinite(b.get("sep", np.nan))
                     else ""))]
        if b.get("in_grid"):
            lines.append((b["ours_criterion"],
                          f"ours q={b['ours_q']:g}  {b['ours_mean']:+.3f}"
                          + (f"  {b['behind']:.1f} hw behind"
                             if np.isfinite(b.get("behind", np.nan)) else "")))
        else:
            lines.append((None, "ours: not in this grid"))
        items = []
        for crit, txt in lines:
            col = fk.CRIT.get(crit, fk.MUTED)
            lead = TextArea(GLYPH.get(fk.CRIT_MARKER.get(crit, "o"), "\u25cf"),
                            textprops=dict(size=BOX_FONT, color=col))
            body = TextArea(txt, textprops=dict(size=BOX_FONT, color=fk.INK))
            items.append(HPacker(children=[lead, body], sep=2.5, pad=0,
                                 align="baseline"))
        box = AnchoredOffsetbox(loc=BOX_CORNER, frameon=True, borderpad=0.28, pad=0.22,
                                child=VPacker(children=items, sep=1.4, pad=0))
        box.patch.set(facecolor="white", edgecolor="#d5d9dd", linewidth=0.6, alpha=0.9)
        a.add_artist(box)
    return b | dict(series=s, metric=metric)


def legend_handles():
    h = []
    for c in CRITS:
        col, mk, lw, _z = fk.crit_style(c, HL)
        h.append(Line2D([], [], color=col, marker=mk, lw=lw, markersize=4.0,
                        label=CRIT_LABEL.get(c, c) + (" (control)" if c == "random" else "")))
    h.append(Line2D([], [], color=fk.C["boost_full"], ls=fk.DASH["boost_full"],
                    label="XGBoost-base (zero)"))
    if SHOW_RING:
        h.append(Line2D([], [], color=fk.INK, marker="o", ls="none",
                        markerfacecolor="none", markersize=9,
                        label="reported configuration"))
    if BAND_CRITS:
        h.append(Line2D([], [], color=fk.MUTED, lw=6, alpha=BAND_ALPHA * 1.6,
                        label=f"{CI_LEVEL:.0%} CI over splits"))
    if SHOW_FLOOR:
        h.append(Line2D([], [], color=fk.MUTED, lw=1.2, ls="none", marker="|",
                        markersize=8,
                        label="I = smallest difference this grid resolves"))
    return h


w = FIG_WIDTH_IN / len(PANELS)
fig, ax = plt.subplots(1, len(PANELS), squeeze=False,
                       figsize=(w * len(PANELS), w * ASPECT))
PAIR = []
for i, (ds, regime) in enumerate(PANELS):
    a = ax[0][i]
    b = draw_panel(a, ds, regime)
    if b:
        PAIR.append(b)
    mtex = METRIC_TEX.get(MOF.get(ds), MOF.get(ds))
    a.set_title(f"{DS_LABEL.get(ds, ds)} - {RG_LABEL.get(regime, regime)}  ({mtex})",
                fontsize=TITLE_FONT, loc="center", pad=TITLE_PAD)
    if i == 0:
        a.set_ylabel(Y_LABEL, fontsize=YLABEL_FONT)
    if PANEL_LETTERS:
        a.annotate(f"({LETTERS[i]})", xy=(0, 1), xycoords="axes fraction",
                   xytext=LETTER_OFFSET, textcoords="offset points",
                   fontweight="bold", fontsize=8.5, ha="left", va="bottom",
                   annotation_clip=False)

fk.figlegend(fig, legend_handles(), ncol=LEGEND_NCOL, pad_in=LEGEND_PAD_IN,
             fontsize=LEGEND_FONT, handlelength=1.8, columnspacing=1.1)
fk.savefig(fig, "quantile_pair", FIG_DIR, SAVE_FIGS, formats=FIG_FORMATS)
plt.show()


In [ ]:
# ============================ THE NUMBERS, AS TEXT ===========================
# One long table per split, for writing the appendix from. Per panel and per cell of
# the grid (q=0 is ONE cell, the complete graph, whatever criteria were written there):
#   K         odorants that carry messages, mean over folds
#   d_base    cell minus XGBoost-base, paired inside each split (+ = cell ahead)
#   d_ours    reported cell minus this cell, paired (+ = reported cell ahead)
#   p_holm    paired t of d_ours, Holm over the grid of that panel
from scipy import stats as _st


def _holm(p):
    p = pd.Series(p, dtype=float)
    ok = p.dropna().sort_values()
    run, adj = 0.0, {}
    for i, (k, v) in enumerate(ok.items()):
        run = max(run, min(1.0, (len(ok) - i) * v))
        adj[k] = run
    return pd.Series({k: adj.get(k, np.nan) for k in p.index}, dtype=float)


def _num(d, metric):
    return d.assign(**{metric: pd.to_numeric(d[metric], errors="coerce")}).dropna(
        subset=[metric])


def _cell_of(c, q):
    return ("complete graph", 0.0) if float(q) <= 0 else (c, float(q))


def _vs(a, b, metric):
    sh = a.dropna().index.intersection(b.dropna().index)
    d = qg.sign_of(metric) * (a.loc[sh] - b.loc[sh])
    m, hw, n = qg.ag.ci(list(d), CI_LEVEL)
    p = (float(_st.ttest_1samp(d, 0.0).pvalue)
         if n >= 3 and float(d.std(ddof=1)) > 0 else np.nan)
    return m, hw, p


def numbers_table(split):
    fr = qg.load(root=ROOT_DIR, dataset=DATASET, regime=REGIME, mol_source=MOL_SOURCE,
                 combo=COMBO, split=split, seeds=SEEDS, gnn_regime=GNN_REGIME)
    out = []
    for ds, regime in PANELS:
        sub, metric = fr[fr.series == f"{ds} / {regime}"], MOF.get(ds)
        if sub.empty or metric is None:
            continue
        g = _num(qg.graph_rows(sub), metric)
        g = g.assign(cell=[_cell_of(c, q) for c, q in zip(g.criterion, g["quantile"])])
        W = (qg.ag.fold_means(g, metric, ["cell"])
             .groupby(["cell", "fold"])[metric].mean().unstack("cell"))
        B = (qg.ag.fold_means(_num(sub[sub.criterion == qg.BOOST], metric), metric)
             .set_index("fold")[metric])
        K = g.groupby("cell")["K"].mean() if "K" in g.columns else pd.Series(dtype=float)
        crit0, q0 = qg.PAPER_POINT[ds]
        key = _cell_of(crit0, q0)
        rows = []
        for c in W.columns:
            db, hb, _ = _vs(W[c], B, metric)
            do, ho, po = (np.nan, np.nan, np.nan) if c == key else _vs(W[key], W[c], metric)
            rows.append(dict(split=split, panel=f"{ds}/{RG_LABEL.get(regime, regime)}",
                             criterion=c[0] + (" *REPORTED*" if c == key else ""),
                             q=c[1], K=K.get(c, np.nan), d_base=db, hw_base=hb,
                             d_ours=do, hw_ours=ho, p=po))
        t = pd.DataFrame(rows)
        t = t.assign(p_holm=_holm(t["p"]).values)
        out.append(t.sort_values(["q", "criterion"]))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


pd.set_option("display.width", 250)
for _split in ("val", "test"):
    T = numbers_table(_split)
    print(f"\n######## split = {_split}   metric: AUROC on M2OR")
    print(T.drop(columns="split").round(4).to_string(index=False))
